# Generate Answer Sheets for Each Student

This workflow creates personalized answer sheets for each student by:
1. Reading student information from an Excel file
2. Loading a template Word document
3. Replacing placeholders with student details (Name, Student ID, Class)
4. Converting each personalized document to PDF
5. Combining all PDFs into a single file
6. Cleaning up intermediate files

In [5]:
prefix = "VTC Test"
name_list = f"../sample/{prefix} Name List.xlsx"
answer_sheet = f"../sample/{prefix} Answer Sheet.docx" 

In [6]:
import pandas as pd

df = pd.read_excel(name_list)
df.head()


,Name,Student,Class
0,Peter,123456789,A
1,Mary,987654321,B
2,John,234567890,C
3,Susan,345678912,D


In [7]:
from docx import Document
from pypdf import PdfWriter, PdfReader
import subprocess
import os

# Create individual PDFs for each student
individual_pdfs = []

for index, row in df.iterrows():
    # Load the template
    doc = Document(answer_sheet)
    
    # Replace placeholders in all paragraphs (preserving formatting)
    for paragraph in doc.paragraphs:
        for run in paragraph.runs:
            if 'Name:' in run.text:
                run.text = run.text.replace('Name:', f"Name: {row['Name']}")
            if 'Student ID:' in run.text:
                run.text = run.text.replace('Student ID:', f"Student ID: {row['Student']}")
            if 'Class:' in run.text:
                run.text = run.text.replace('Class:', f"Class: {row['Class']}")
    
    # Replace placeholders in tables
    for table in doc.tables:
        for row_table in table.rows:
            for cell in row_table.cells:
                for paragraph in cell.paragraphs:
                    for run in paragraph.runs:
                        if 'Name:' in run.text:
                            run.text = run.text.replace('Name:', f"Name: {row['Name']}")
                        if 'Student ID:' in run.text:
                            run.text = run.text.replace('Student ID:', f"Student ID: {row['Student']}")
                        if 'Class:' in run.text:
                            run.text = run.text.replace('Class:', f"Class: {row['Class']}")
    
    # Save the modified docx
    docx_filename = f"../data/{prefix} Answer Sheet - {row['Name']}.docx"
    doc.save(docx_filename)
    
    # Convert to PDF using LibreOffice
    pdf_filename = f"../data/{prefix} Answer Sheet - {row['Name']}.pdf"
    result = subprocess.run([
        'libreoffice', '--headless', '--convert-to', 'pdf', 
        '--outdir', '../data', docx_filename
    ], capture_output=True, text=True)
    
    individual_pdfs.append(pdf_filename)
    print(f"Created PDF for {row['Name']}: {pdf_filename}")

# Merge all PDFs
if individual_pdfs:
    writer = PdfWriter()
    
    for pdf_file in individual_pdfs:
        reader = PdfReader(pdf_file)
        for page in reader.pages:
            writer.add_page(page)
    
    # Write merged PDF
    output_file = f"../data/{prefix} Answer Sheets Combined.pdf"
    with open(output_file, 'wb') as output:
        writer.write(output)
    
    print(f"\nCombined PDF saved to: {output_file}")
    print(f"Total students processed: {len(df)}")
    
    # Clean up intermediate files
    print("\nCleaning up intermediate files...")
    for index, row in df.iterrows():
        docx_file = f"../data/{prefix} Answer Sheet - {row['Name']}.docx"
        pdf_file = f"../data/{prefix} Answer Sheet - {row['Name']}.pdf"
        if os.path.exists(docx_file):
            os.remove(docx_file)
        if os.path.exists(pdf_file):
            os.remove(pdf_file)
    print("Cleanup complete!")

Created PDF for Peter: ../data/VTC Test Answer Sheet - Peter.pdf
Created PDF for Mary: ../data/VTC Test Answer Sheet - Mary.pdf
Created PDF for John: ../data/VTC Test Answer Sheet - John.pdf
Created PDF for Susan: ../data/VTC Test Answer Sheet - Susan.pdf

Combined PDF saved to: ../data/VTC Test Answer Sheets Combined.pdf
Total students processed: 4

Cleaning up intermediate files...
Cleanup complete!
